In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

dark_bg = '#1e1e2e'
colors = ['#4e9fd4','#d4824e','#4ed484','#d44e4e','#9b4ed4',
          '#d4c44e','#4ec4d4','#d44e9b','#7ed44e','#4e6bd4']

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
conn = sqlite3.connect('db.sqlite3')

query = """
    SELECT
        oi.id          AS "Order Item ID",
        oi.quantity    AS "Quantity",
        oi.order_id    AS "Order ID",
        oi.product_id  AS "Product ID",
        p.name         AS "Product Name",
        p.price        AS "Product Price",
        o.datetime     AS "Order Datetime"
    FROM restaurant_orderitem oi
    JOIN restaurant_product p ON oi.product_id = p.id
    JOIN restaurant_order   o ON oi.order_id   = o.id
"""

df = pd.read_sql_query(query, conn)
conn.close()

df.to_csv('restaurant_data.csv', index=False)
df = pd.read_csv('restaurant_data.csv', parse_dates=['Order Datetime'])

print(f'Total rows: {len(df)}')
df.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top10_qty = (
    df.groupby('Product Name')['Quantity']
    .sum()
    .nlargest(10)
)

fig, ax = plt.subplots(figsize=(8, 6), facecolor=dark_bg)
ax.set_facecolor(dark_bg)
ax.pie(
    top10_qty.values,
    labels=top10_qty.index,
    autopct=lambda p: f'{p:.1f}% ({int(round(p * top10_qty.sum() / 100))})',
    colors=colors,
    startangle=90,
    pctdistance=0.75,
    textprops={'color': 'white', 'fontsize': 8}
)
ax.set_title('Top 10 positions in menu by quantity', color='white', fontsize=13, pad=15)
ax.legend(['Quantity'], loc='center left', labelcolor='white',
          facecolor=dark_bg, edgecolor='gray', fontsize=9)
plt.tight_layout()
plt.show()

print(top10_qty.to_string())

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df['Item Price'] = df['Product Price'] * df['Quantity']

top10_price = (
    df.groupby('Product Name')['Item Price']
    .sum()
    .nlargest(10)
)

fig, ax = plt.subplots(figsize=(8, 6), facecolor=dark_bg)
ax.set_facecolor(dark_bg)
ax.pie(
    top10_price.values,
    labels=top10_price.index,
    autopct=lambda p: f'{p:.1f}% (\u00a3{int(round(p * top10_price.sum() / 100))})',
    colors=colors,
    startangle=90,
    pctdistance=0.75,
    textprops={'color': 'white', 'fontsize': 8}
)
ax.set_title('Top 10 positions in menu by revenue (\u00a3)', color='white', fontsize=13, pad=15)
ax.legend(['Item Price'], loc='center left', labelcolor='white',
          facecolor=dark_bg, edgecolor='gray', fontsize=9)
plt.tight_layout()
plt.show()

print(top10_price.round(2).to_string())

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df['Order Hour'] = df['Order Datetime'].dt.hour

income_by_hour = (
    df.groupby('Order Hour')['Item Price']
    .sum()
    .reindex(range(24), fill_value=0)
)

fig, ax = plt.subplots(figsize=(9, 5), facecolor=dark_bg)
ax.set_facecolor(dark_bg)
ax.bar(income_by_hour.index, income_by_hour.values, color='#4e9fd4', width=0.7)
ax.set_title('Profit by Order Hour', color='white', fontsize=13)
ax.set_xlabel('Order Hour', color='white', fontsize=10)
ax.set_ylabel('Total Income (\u00a3)', color='white', fontsize=10)
ax.tick_params(colors='white')
ax.spines[:].set_visible(False)
plt.tight_layout()
plt.show()

print(income_by_hour[income_by_hour > 0].round(2).to_string())

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df['Order Day'] = df['Order Datetime'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

income_by_day = (
    df.groupby('Order Day')['Item Price']
    .sum()
    .reindex(day_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(9, 5), facecolor=dark_bg)
ax.set_facecolor(dark_bg)
ax.bar(income_by_day.index, income_by_day.values, color='#4e9fd4', width=0.6)
ax.set_title('Profit by Day of Week', color='white', fontsize=13)
ax.set_xlabel('Day of Week', color='white', fontsize=10)
ax.set_ylabel('Total Income (\u00a3)', color='white', fontsize=10)
ax.tick_params(colors='white')
ax.spines[:].set_visible(False)
plt.tight_layout()
plt.show()

print(income_by_day.round(2).to_string())